# **Qwen2.5-omni**

## 1.환경준비

### (1) 라이브러리 설치

In [ ]:
!pip -q install -U "transformers>=4.52.0" accelerate qwen-omni-utils[decord] soundfile

### (2) 라이브러리 로딩

In [ ]:
import torch
from transformers import Qwen2_5OmniForConditionalGeneration, Qwen2_5OmniProcessor
from qwen_omni_utils import process_mm_info


## 2.Qwen 사용해보기

### (1) 모델 다운로드
* 모델 다운로드 : 5~8분

In [3]:
model_id = "Qwen/Qwen2.5-Omni-3B"  # 경량
model = Qwen2_5OmniForConditionalGeneration.from_pretrained(
    model_id, torch_dtype=torch.float16, device_map="auto"
)
processor = Qwen2_5OmniProcessor.from_pretrained(model_id)

`torch_dtype` is deprecated! Use `dtype` instead!
c:\Users\USER\anaconda3\envs\aivle_pytorch\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\USER\.cache\huggingface\hub\models--Qwen--Qwen2.5-Omni-3B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Unrecognized keys in `rope_scaling` for 'rope_type'='def

### (2) 모델 사용하기

* 파일 준비

In [ ]:
from google.colab import files
uploaded = files.upload()   # 로컬 PC에서 beach_children.jpg 선택

* 모델 사용

In [4]:
USE_AUDIO_IN_VIDEO = False  # 영상의 내부 오디오까지 쓸지 여부
USE_AUDIO = False

conversation = [
    {"role":"system","content":[{"type":"text","text":"You are a helpful assistant."}]},
    {"role":"user","content":[
        {"type":"image","image":"./data/beach_children.jpg"},
        {"type":"text","text":"사람 수와 상황을 설명해줘."}
    ]}
]

text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
audios, images, videos = process_mm_info(conversation, use_audio_in_video=USE_AUDIO_IN_VIDEO)

inputs = processor(
    text=text, audio=audios, images=images, videos=videos,
    return_tensors="pt", padding=True, use_audio_in_video=USE_AUDIO_IN_VIDEO
).to(model.device)

gen_kwargs = dict(
    **inputs,
    return_audio=USE_AUDIO,
    use_audio_in_video=False,
    max_new_tokens=256,
)

with torch.inference_mode():
    out = model.generate(**gen_kwargs)

# 출력 분기 처리
if USE_AUDIO:
    text_ids, audio = out            # 음성까지 요청한 경우: (text_ids, audio) 튜플
else:
    text_ids = out                   # 텍스트만 요청한 경우: 텐서 1개

print(processor.batch_decode(text_ids, skip_special_tokens=True)[0])

system
You are a helpful assistant.
user
사람 수와 상황을 설명해줘.
assistant
이 사진에는 총 3명의 사람이 보입니다. 

1. 왼쪽에 앉아 있는 두 명의 어린이가 모래를 모으고 있습니다.
2. 오른쪽에 서 있는 어린이가 물을 들고 있습니다.

모든 어린이들은 바닷가에서 놀고 있는 상황입니다.


### (3) 실습
* 다양한 이미지를 다운받아, 모델에 입력하고, 사용해 봅시다.

In [5]:
USE_AUDIO_IN_VIDEO = False  # 영상의 내부 오디오까지 쓸지 여부
USE_AUDIO = False

conversation = [
    {"role":"system","content":[{"type":"text","text":"You are a helpful assistant."}]},
    {"role":"user","content":[
        {"type":"image","image":"./data/dog.jpg"},
        {"type":"text","text":"어떤 동물이 보이는지 설명해줘."}
    ]}
]

text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
audios, images, videos = process_mm_info(conversation, use_audio_in_video=USE_AUDIO_IN_VIDEO)

inputs = processor(
    text=text, audio=audios, images=images, videos=videos,
    return_tensors="pt", padding=True, use_audio_in_video=USE_AUDIO_IN_VIDEO
).to(model.device)

gen_kwargs = dict(
    **inputs,
    return_audio=USE_AUDIO,
    use_audio_in_video=False,
    max_new_tokens=256,
)

with torch.inference_mode():
    out = model.generate(**gen_kwargs)

# 출력 분기 처리
if USE_AUDIO:
    text_ids, audio = out            # 음성까지 요청한 경우: (text_ids, audio) 튜플
else:
    text_ids = out                   # 텍스트만 요청한 경우: 텐서 1개

print(processor.batch_decode(text_ids, skip_special_tokens=True)[0])

system
You are a helpful assistant.
user
어떤 동물이 보이는지 설명해줘.
assistant
이 사진에는 갈색의 개가 보입니다. 개는 빨간색의 옷을 입고 있으며, 흰색의 꼬리를 가지고 있습니다.


In [6]:
USE_AUDIO_IN_VIDEO = False  # 영상의 내부 오디오까지 쓸지 여부
USE_AUDIO = False

conversation = [
    {"role":"system","content":[{"type":"text","text":"You are a helpful assistant."}]},
    {"role":"user","content":[
        {"type":"image","image":"./data/cat-bird_wide-85ce4b8383b9440d3ff03413cdd913513e9737bf-s4-c85.jpg"},
        {"type":"text","text":"어떤 상황인지 설명해줘."}
    ]}
]

text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
audios, images, videos = process_mm_info(conversation, use_audio_in_video=USE_AUDIO_IN_VIDEO)

inputs = processor(
    text=text, audio=audios, images=images, videos=videos,
    return_tensors="pt", padding=True, use_audio_in_video=USE_AUDIO_IN_VIDEO
).to(model.device)

gen_kwargs = dict(
    **inputs,
    return_audio=USE_AUDIO,
    use_audio_in_video=False,
    max_new_tokens=256,
)

with torch.inference_mode():
    out = model.generate(**gen_kwargs)

# 출력 분기 처리
if USE_AUDIO:
    text_ids, audio = out            # 음성까지 요청한 경우: (text_ids, audio) 튜플
else:
    text_ids = out                   # 텍스트만 요청한 경우: 텐서 1개

print(processor.batch_decode(text_ids, skip_special_tokens=True)[0])

system
You are a helpful assistant.
user
어떤 상황인지 설명해줘.
assistant
이 사진은 고양이가 새를 잡은 상황을 보여주고 있습니다. 고양이는 녹색 식물 사이에 서 있으며, 새를 입에 들고 있습니다. 이는 고양이가 자연에서 사냥하는 모습을 보여주는 듯합니다.
